In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

In [ ]:
train_csv = "/kaggle/input/digit-recognizer/train.csv"
test_csv = "/kaggle/input/digit-recognizer/test.csv"

df = pd.read_csv(train_csv)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

x_train = train_df.iloc[:, 1:].values.astype(np.float32) / 255.0
y_train = train_df.iloc[:, 0].values.astype(np.int64)
x_val = val_df.iloc[:, 1:].values.astype(np.float32) / 255.0
y_val = val_df.iloc[:, 0].values.astype(np.int64)

x_train, y_train, x_val, y_val = map(torch.tensor, (x_train, y_train, x_val, y_val))

In [ ]:
class DigitDataset(Dataset):
    def __init__(self, X, Y):
        self.X, self.Y = X, Y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

BATCH_SIZE = 128
train_dl = DataLoader(DigitDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(DigitDataset(x_val, y_val), batch_size=BATCH_SIZE)

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.out = nn.Linear(128, 10)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        x = self.drop(F.relu(self.bn1(self.fc1(x))))
        x = self.drop(F.relu(self.bn2(self.fc2(x))))
        x = self.drop(F.relu(self.bn3(self.fc3(x))))
        return self.out(x)

model = MNISTNet().to(device)
print(model)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

 
    model.eval()
    with torch.no_grad():
        val_preds = []
        val_labels = []
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(yb.cpu().numpy())
        acc = np.mean(np.array(val_preds) == np.array(val_labels))

    print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {total_loss/len(train_dl):.4f} | Val Acc: {acc*100:.2f}%")

print("\n Training Complete!")

In [ ]:
print(f"Final Validation Accuracy: {acc*100:.2f}%")